# 양자화 (Quantization) 정리

## 양자화란?

양자화(Quantization)는 **딥러닝 모델의 파라미터(가중치 및 활성값)** 를 고정 소수점 숫자(예: int8, int4 등)로 변환하여 **모델 크기 및 추론 속도를 줄이는 기술**입니다.

---

##  Hugging Face에서 지원하는 주요 양자화 방식

| 방식 | 설명 | 장점 | 단점 |
|------|------|------|------|
| **8-bit 양자화** | `load_in_8bit=True` | - 메모리 절약<br>- 대부분의 GPU에서 안정적 | - 약간의 정확도 손실 |
| **4-bit 양자화 (QLoRA)** | `load_in_4bit=True`<br>with `BitsAndBytesConfig` | - 극단적인 메모리 절감<br>- LoRA와 함께 사용 가능 | - 일부 모델/하드웨어에서 불안정 |
| **Dynamic Quantization** | 추론 시 실시간 변환 | - 간단한 적용 | - 낮은 효율 |
| **Static Quantization** | 미리 정량화 테이블 생성 | - 효율적 | - 사전 분석 필요 |

---

##  주요 라이브러리: `bitsandbytes`

```bash
pip install bitsandbytes


NVIDIA GPU에서 양자화된 모델을 효율적으로 실행 가능

Hugging Face transformers와 통합

| 상황                           | 추천 양자화                     |
| ---------------------------- | -------------------------- |
| **로컬 GPU 메모리가 작을 때**         | 4bit (QLoRA)               |
| **정확도 손실을 최소화하면서 압축하고 싶을 때** | 8bit                       |
| **빠른 추론이 필요할 때**             | 8bit / Static Quantization |
| **LoRA 튜닝과 병행할 때**           | 4bit + QLoRA               |


🚧 주의사항
일부 모델은 4bit를 완전히 지원하지 않을 수 있음

CPU 환경에서는 bitsandbytes가 동작하지 않음 (GPU 필수)

load_in_8bit, load_in_4bit 사용 시 trust_remote_code=True 옵션 필요할 수 있음

| 옵션                                 | 설명                                                                                                                       |
| ---------------------------------- | ------------------------------------------------------------------------------------------------------------------------ |
| `load_in_4bit=True`                | 모델을 **4bit 양자화 모드**로 로드합니다. <br>메모리를 극도로 절약할 수 있고, QLoRA 튜닝과 함께 사용됩니다.                                                   |
| `bnb_4bit_quant_type="nf4"`        | 4bit 양자화의 방식 선택:<br>  - `"nf4"`: **NormalFloat4**, 더 정밀한 표현 가능<br>  - `"fp4"`: 4bit floating-point 형식 (일반적으로 NF4가 더 정밀함) |
| `bnb_4bit_use_double_quant=True`   | **이중 양자화(double quantization)** 사용 여부<br> - 가중치를 두 번 양자화하여 더 높은 압축률 및 정밀도 보존<br> - 보통 `True`가 성능 좋음                      |
| `bnb_4bit_compute_dtype="float16"` | 연산 시 사용할 데이터 타입 <br> - `"float16"`: 연산 성능이 빠르고 대부분 GPU에서 지원 <br> - `"bfloat16"`: 일부 GPU(A100, H100 등)에서 정밀도 더 높음         |


###  양자화(Quantization)의 원리

####  1. 개념 요약

양자화는 딥러닝 모델의 **가중치(weights)** 또는 **활성값(activations)** 을 
32-bit float → 8-bit / 4-bit 정수로 변환하여,
- 모델 크기를 줄이고
- 추론 속도와 효율을 높이며
- 메모리 사용을 절감하는 기술입니다.

---

####  2. 정수 표현 예시 (8bit)

| 값 종류 | 32bit (float32) | 8bit (int8) |
|----------|------------------|--------------|
| 저장 크기 | 4 bytes | 1 byte |
| 표현 범위 | $[-3.4 \times 10^{38}, 3.4 \times 10^{38}]$ | [-128, 127] |
| 정밀도 | 매우 높음 | 낮음 (압축) |

---

####  3. 양자화 종류

| 종류 | 설명 | 적용 대상 | 사용 예시 |
|------|------|-----------|-----------|
| **Post-Training Quantization (PTQ)** | 학습 후 양자화 | 가중치 & 활성값 | 대부분의 추론 최적화 |
| **Quantization-Aware Training (QAT)** | 학습 중 양자화 고려 | 모델 전체 | 정확도 유지 중요할 때 |
| **Dynamic Quantization** | 실행 시 양자화 적용 | 활성값 | CPU 기반 추론 |
| **Static Quantization** | 사전 측정된 범위로 양자화 | 가중치 & 활성값 | 빠른 추론에 적합 |

---

##  4. Hugging Face에서의 적용 예

```python
from transformers import BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,  # 또는 load_in_4bit=True
)


### PTQ (Post-Training Quantization) : 기존 모델을 바꿔치기만 하면 됨

- 양자화 대상: weight, activation
- 손실 있음 (especially low-bit, ex. 4bit)
- Hugging Face의 BitsAndBytes가 대표 예

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline, BitsAndBytesConfig


# 4bit 양자화 설정
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype="float16"
)
'''
# 8bit 양자화 설정
bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,                    #  8bit 로드
    llm_int8_threshold=6.0,               # LayerNorm보다 작은 weight만 quantize
    llm_int8_skip_modules=None,           # 특정 모듈은 quantize 생략 가능
    llm_int8_enable_fp32_cpu_offload=True # 일부 연산을 CPU로 오프로드
)
'''

# 모델 및 토크나이저 로딩 (4bit 양자화 적용)
model = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Llama-3.1-8B-Instruct",
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)
tokenizer = AutoTokenizer.from_pretrained(
    "meta-llama/Llama-3.1-8B-Instruct",
    trust_remote_code=True
)

# pipeline 구성
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer
)

# 예시 추론
output = pipe("What is the capital of France?", max_new_tokens=20)
print(output[0]["generated_text"])